# Импорт модулей

In [65]:
import os
import pandas as pd
import numpy as np
from itertools import combinations

from config import PERIODS, METHODS, CORR_LOWER, CORR_UPPER, TRADING_DAYS
from utils import (
    equal_weight_metrics, normalize_pair, normalize_triple, normalize_quad,
    ensure_dirs
)

ensure_dirs(['results/tables', 'data'])

# Определение списка тикеров

In [67]:
top21_file = 'results/tables/top21_tickers.csv'
if not os.path.exists(top21_file):
    raise FileNotFoundError(f"{top21_file} не найден. Сначала запустите 01_filtering_and_metrics.")
tickers = pd.read_csv(top21_file)['ticker'].tolist()
print(f"Загружено {len(tickers)} акций:", tickers)

Загружено 21 акций: ['SPBE', 'MRKV', 'LENT', 'RBCM', 'OZON', 'STSBP', 'MRKY', 'RTSB', 'MRKU', 'SBER', 'RTSBP', 'TGKN', 'RZSB', 'SBERP', 'ETLN', 'RNFT', 'MTSS', 'SVAV', 'RENI', 'AFLT', 'RTGZ']


# Функция для поиска портфелей (n=2,3,4) с корреляционным условием

In [69]:
def find_portfolios(n, period_name, method):
    price_file = os.path.join('data', f"sortino_tickers_prices_{period_name}_{method}.csv")
    df_prices = pd.read_csv(price_file, index_col=0, parse_dates=True)
    returns = df_prices.pct_change().dropna()
    corr = returns.corr()
    
    valid = []
    for combo in combinations(tickers, n):
        if any(t not in corr.columns for t in combo):
            continue
        ok = True
        for t1, t2 in combinations(combo, 2):
            r = corr.loc[t1, t2]
            if not (CORR_LOWER <= r <= CORR_UPPER):
                ok = False
                break
        if ok:
            valid.append(combo)
    return valid

# Расчёт равновзвешенных метрик для найденных портфелей

In [71]:
def process_portfolios(portfolios, n, period_name, method):
    price_file = os.path.join('data', f"sortino_tickers_prices_{period_name}_{method}.csv")
    df_prices = pd.read_csv(price_file, index_col=0, parse_dates=True)
    returns = df_prices.pct_change().dropna()
    mean_ret = returns.mean()
    cov = returns.cov()
    rf = PERIODS[period_name]['rf']
    results = []
    for combo in portfolios:
        ret, risk, sharpe = equal_weight_metrics(list(combo), mean_ret, cov, rf)
        record = {f'ticker{i+1}': ticker for i, ticker in enumerate(combo)}
        record['annual_return'] = ret
        record['annual_risk'] = risk
        record['sharpe'] = sharpe
        results.append(record)
    return pd.DataFrame(results)

# Обработка всех периодов и методов для пар (n=2)

In [73]:
pairs_dict = {}
for period_name in PERIODS.keys():
    for method in METHODS:
        portfolios = find_portfolios(2, period_name, method)
        if not portfolios:
            print(f"{period_name} {method}: пар не найдено")
            continue
        df = process_portfolios(portfolios, 2, period_name, method)
        out_file = os.path.join('results/tables', f"pairs_corr_{CORR_LOWER}_to_{CORR_UPPER}_{period_name}_{method}.csv")
        df.to_csv(out_file, index=False)
        pairs_dict[(period_name, method)] = df
        print(f"{period_name} {method}: найдено {len(df)} пар")

2023_2024 inner: найдено 116 пар
2023_2024 ffill: найдено 117 пар
2024_2025 inner: найдено 90 пар
2024_2025 ffill: найдено 101 пар


# Поиск общих пар для всех четырёх наборов

In [75]:
pair_sets = {}
for period_name in PERIODS.keys():
    for method in METHODS:
        df = pairs_dict.get((period_name, method))
        if df is not None and not df.empty:
            df['pair_key'] = df.apply(lambda r: normalize_pair(r['ticker1'], r['ticker2']), axis=1)
            pair_sets[(period_name, method)] = set(df['pair_key'])

if len(pair_sets) == 4:
    common_keys = set.intersection(*pair_sets.values())
    print(f"Общих пар во всех 4 наборах: {len(common_keys)}")
    if common_keys:
        sample_df = pairs_dict[('2023_2024', 'inner')]
        common_data = []
        for key in common_keys:
            row = sample_df[sample_df['pair_key'] == key].iloc[0]
            common_data.append({
                'ticker1': key[0],
                'ticker2': key[1],
                'annual_return': row['annual_return'],
                'annual_risk': row['annual_risk'],
                'sharpe': row['sharpe']
            })
        df_common_pairs = pd.DataFrame(common_data)
        df_common_pairs.to_csv('results/tables/common_pairs_all_datasets.csv', index=False)
        print("Список общих пар сохранён в results/tables/common_pairs_all_datasets.csv")
else:
    print("Не хватает данных для поиска общих пар.")

Общих пар во всех 4 наборах: 51
Список общих пар сохранён в results/tables/common_pairs_all_datasets.csv


# Тройки (n=3) 

In [77]:
triples_dict = {}
for period_name in PERIODS.keys():
    for method in METHODS:
        portfolios = find_portfolios(3, period_name, method)
        if not portfolios:
            print(f"{period_name} {method}: троек не найдено")
            continue
        df = process_portfolios(portfolios, 3, period_name, method)
        out_file = os.path.join('results/tables', f"triples_corr_{CORR_LOWER}_to_{CORR_UPPER}_{period_name}_{method}.csv")
        df.to_csv(out_file, index=False)
        triples_dict[(period_name, method)] = df
        print(f"{period_name} {method}: найдено {len(df)} троек")

triple_sets = {}
for period_name in PERIODS.keys():
    for method in METHODS:
        df = triples_dict.get((period_name, method))
        if df is not None and not df.empty:
            df['triple_key'] = df.apply(lambda r: normalize_triple(r['ticker1'], r['ticker2'], r['ticker3']), axis=1)
            triple_sets[(period_name, method)] = set(df['triple_key'])

if len(triple_sets) == 4:
    common_triple_keys = set.intersection(*triple_sets.values())
    print(f"\nОбщих троек во всех 4 наборах: {len(common_triple_keys)}")
    if common_triple_keys:
        sample_df = triples_dict[('2023_2024', 'inner')]
        common_data = []
        for key in common_triple_keys:
            row = sample_df[sample_df['triple_key'] == key].iloc[0]
            common_data.append({
                'ticker1': key[0],
                'ticker2': key[1],
                'ticker3': key[2],
                'annual_return': row['annual_return'],
                'annual_risk': row['annual_risk'],
                'sharpe': row['sharpe']
            })
        df_common_triples = pd.DataFrame(common_data)
        df_common_triples.to_csv('results/tables/common_triples_all_datasets.csv', index=False)
        print("Список общих троек сохранён в results/tables/common_triples_all_datasets.csv")
else:
    print("Не хватает данных для поиска общих троек.")

2023_2024 inner: найдено 239 троек
2023_2024 ffill: найдено 240 троек
2024_2025 inner: найдено 121 троек
2024_2025 ffill: найдено 161 троек

Общих троек во всех 4 наборах: 17
Список общих троек сохранён в results/tables/common_triples_all_datasets.csv


# Четвёрки (n=4)

In [79]:
quadruples_dict = {}
for period_name in PERIODS.keys():
    for method in METHODS:
        portfolios = find_portfolios(4, period_name, method)
        if not portfolios:
            print(f"{period_name} {method}: четвёрок не найдено")
            continue
        df = process_portfolios(portfolios, 4, period_name, method)
        out_file = os.path.join('results/tables', f"quadruples_corr_{CORR_LOWER}_to_{CORR_UPPER}_{period_name}_{method}.csv")
        df.to_csv(out_file, index=False)
        quadruples_dict[(period_name, method)] = df
        print(f"{period_name} {method}: найдено {len(df)} четвёрок")

quad_sets = {}
for period_name in PERIODS.keys():
    for method in METHODS:
        df = quadruples_dict.get((period_name, method))
        if df is not None and not df.empty:
            df['quad_key'] = df.apply(lambda r: normalize_quad(r['ticker1'], r['ticker2'], r['ticker3'], r['ticker4']), axis=1)
            quad_sets[(period_name, method)] = set(df['quad_key'])

if len(quad_sets) == 4:
    common_quad_keys = set.intersection(*quad_sets.values())
    print(f"\nОбщих четвёрок во всех 4 наборах: {len(common_quad_keys)}")
    if common_quad_keys:
        sample_df = quadruples_dict[('2023_2024', 'inner')]
        common_data = []
        for key in common_quad_keys:
            row = sample_df[sample_df['quad_key'] == key].iloc[0]
            common_data.append({
                'ticker1': key[0],
                'ticker2': key[1],
                'ticker3': key[2],
                'ticker4': key[3],
                'annual_return': row['annual_return'],
                'annual_risk': row['annual_risk'],
                'sharpe': row['sharpe']
            })
        df_common_quads = pd.DataFrame(common_data)
        df_common_quads.to_csv('results/tables/common_quadruples_all_datasets.csv', index=False)
        print("Список общих четвёрок сохранён в results/tables/common_quadruples_all_datasets.csv")
else:
    print("Недостаточно данных для поиска общих четвёрок.")

2023_2024 inner: найдено 219 четвёрок
2023_2024 ffill: найдено 210 четвёрок
2024_2025 inner: найдено 55 четвёрок
2024_2025 ffill: найдено 88 четвёрок

Общих четвёрок во всех 4 наборах: 0


In [90]:
print("\nПромежуточные итоги")
for period_name in PERIODS.keys():
    for method in METHODS:
        pairs_file = os.path.join('results/tables', f"pairs_corr_{CORR_LOWER}_to_{CORR_UPPER}_{period_name}_{method}.csv")
        triples_file = os.path.join('results/tables', f"triples_corr_{CORR_LOWER}_to_{CORR_UPPER}_{period_name}_{method}.csv")
        quadruples_file = os.path.join('results/tables', f"quadruples_corr_{CORR_LOWER}_to_{CORR_UPPER}_{period_name}_{method}.csv")
        n_pairs = len(pd.read_csv(pairs_file)) if os.path.exists(pairs_file) else 0
        n_triples = len(pd.read_csv(triples_file)) if os.path.exists(triples_file) else 0
        n_quads = len(pd.read_csv(quadruples_file)) if os.path.exists(quadruples_file) else 0
        print(f"{period_name} {method}: пар={n_pairs}, троек={n_triples}, четвёрок={n_quads}")


Промежуточные итоги
2023_2024 inner: пар=116, троек=239, четвёрок=219
2023_2024 ffill: пар=117, троек=240, четвёрок=210
2024_2025 inner: пар=90, троек=121, четвёрок=55
2024_2025 ffill: пар=101, троек=161, четвёрок=88
